# Silver Layer — `dim_customers` cu SCD Type 2

**Scop:** Construirea dimensiunii `dim_customers` cu **Slowly Changing Dimension Type 2** — păstrăm istoricul complet al modificărilor pe atributele urmărite.

**Atribute urmărite (track):** `segment_code`, `kyc_status`, `county_code`, `is_active`

**Atribute non-track (SCD Type 1 — overwrite):** `email`, `phone`, `address`, `city`, `first_name`, `last_name`

**Coloane SCD Type 2 generate:**
- `customer_sk` — surrogate key auto-incrementat (PK în Silver)
- `valid_from` — timestamp de la care versiunea e activă
- `valid_to` — timestamp până la care versiunea e activă (`NULL` pentru cea curentă)
- `is_current` — flag pentru identificare rapidă (Kimball pattern)

**Algoritm:**
- **Prima rulare** (Silver gol): INSERT toate rândurile cu `is_current = true`
- **Rulări următoare**: MERGE
  - Client nou → INSERT
  - Atribute neschimbate → skip
  - Atribute schimbate → UPDATE versiunea curentă + INSERT versiune nouă
  - Client lipsă din Bronze → soft delete (close versiune curentă)

## 1. Configurare

In [0]:
from datetime import datetime
from pyspark.sql import DataFrame, Window
from pyspark.sql.functions import (
    col, when, lit, current_timestamp, to_date, to_timestamp,
    array, size, concat_ws, sha2, concat,
    coalesce, row_number, monotonically_increasing_id,
    max as spark_max,
    filter as array_filter
)
from pyspark.sql.types import IntegerType, DoubleType, StringType, TimestampType
from delta.tables import DeltaTable

CATALOG     = "banking"
BRONZE      = "bronze"
SILVER      = "silver"
QUARANTINE  = "silver_quarantine"

TARGET_TABLE = f"{CATALOG}.{SILVER}.dim_customers"
QUARANTINE_TABLE = f"{CATALOG}.{QUARANTINE}.qrt_customers"

# Atribute urmarite pentru SCD Type 2
SCD_TRACKED = ["segment_code", "kyc_status", "county_code", "is_active"]

print(f"Target:     {TARGET_TABLE}")
print(f"Quarantine: {QUARANTINE_TABLE}")
print(f"SCD tracked: {SCD_TRACKED}")

Target:     banking.silver.dim_customers
Quarantine: banking.silver_quarantine.qrt_customers
SCD tracked: ['segment_code', 'kyc_status', 'county_code', 'is_active']


## 2. Citire Bronze și pregătire date

In [0]:
df = spark.table(f"{CATALOG}.{BRONZE}.raw_customers")

# Pastram metadata Bronze pentru lineage
if "_batch_id" in df.columns:
    df = df.withColumnRenamed("_batch_id", "source_batch_id")
if "_ingestion_ts" in df.columns:
    df = df.withColumnRenamed("_ingestion_ts", "source_ingestion_ts")
for c in ["_source_system", "_source_file"]:
    if c in df.columns:
        df = df.drop(c)

# Type casting
df = df.withColumn("is_active", col("is_active").cast(IntegerType()))
df = df.withColumn("date_of_birth",   to_date(col("date_of_birth"),   "yyyy-MM-dd"))
df = df.withColumn("kyc_verified_at", to_timestamp(col("kyc_verified_at")))

# Lista de coloane FK valide (verificate fata de tabelele ref Silver)
valid_counties  = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_counties")
                   .select("county_code").collect()]
valid_segments  = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_customer_segments")
                   .select("segment_code").collect()]
valid_kyc       = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_kyc_statuses")
                   .select("status_code").collect()]

print(f"Bronze rows: {df.count()}")
print(f"Refs incarcate: counties={len(valid_counties)}, segments={len(valid_segments)}, kyc={len(valid_kyc)}")

Bronze rows: 500
Refs incarcate: counties=42, segments=4, kyc=4


## 3. Validări → split valid / quarantine

In [0]:
from pyspark.sql.functions import filter as array_filter

validations = [
    ("pk_null",            col("customer_id").isNotNull()),
    ("cnp_null",           col("cnp").isNotNull()),
    ("cnp_length",         (col("cnp").isNull()) | (col("cnp").rlike("^[0-9]{13}$"))),
    ("email_null",         col("email").isNotNull()),
    ("email_format",       (col("email").isNull()) | (col("email").rlike(".+@.+\\..+"))),
    ("dob_null",           col("date_of_birth").isNotNull()),
    ("dob_future",         (col("date_of_birth").isNull()) | (col("date_of_birth") <= current_timestamp().cast("date"))),
    ("gender_invalid",     col("gender").isin(["M", "F"])),
    ("county_invalid",     col("county_code").isin(valid_counties)),
    ("segment_invalid",    col("segment_code").isin(valid_segments)),
    ("kyc_invalid",        col("kyc_status").isin(valid_kyc)),
]

# Construim coloane auxiliare _err_* pentru fiecare regula
error_cols = []
for rule_name, condition in validations:
    col_name = f"_err_{rule_name}"
    df = df.withColumn(col_name, when(~condition, lit(rule_name)))
    error_cols.append(col_name)

# Folosim higher-order function filter() pentru a elimina NULL-urile
df = df.withColumn(
    "error_reasons",
    array_filter(array(*[col(c) for c in error_cols]), lambda x: x.isNotNull())
)

# Cleanup coloane auxiliare
df = df.drop(*error_cols)

df_valid   = df.filter(size(col("error_reasons")) == 0).drop("error_reasons")
df_invalid = df.filter(size(col("error_reasons")) > 0)

n_valid   = df_valid.count()
n_invalid = df_invalid.count()
total     = n_valid + n_invalid
pct       = (n_invalid / total * 100) if total > 0 else 0
print(f"Valid:   {n_valid:>6d}")
print(f"Invalid: {n_invalid:>6d}  ({pct:.1f}% quarantine)")

# Scriem quarantine
if n_invalid > 0:
    (df_invalid
        .withColumn("quarantine_loaded_at", current_timestamp())
        .withColumn("error_reasons_str", concat_ws(", ", col("error_reasons")))
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(QUARANTINE_TABLE))

Valid:      500
Invalid:      0  (0.0% quarantine)


## 4. Hash al atributelor SCD-tracked

Generăm un hash al celor 4 atribute urmărite. Compararea hash-urilor între versiunea curentă din Silver și rândul nou din Bronze ne spune instant dacă a fost vreo modificare.

**De ce hash și nu compare coloană cu coloană?** Mai rapid în Spark (un singur JOIN pe hash) și mai lizibil în cod.

In [0]:
# Construim hash-ul atributelor tracked (pentru detectarea modificarilor)
df_valid = df_valid.withColumn(
    "scd_hash",
    sha2(concat_ws("|", *[coalesce(col(c).cast("string"), lit("__NULL__")) for c in SCD_TRACKED]), 256)
)

print("Coloane in df_valid dupa hash:")
df_valid.printSchema()

Coloane in df_valid dupa hash:
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- cnp: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- county_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- address: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- segment_code: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- kyc_verified_at: timestamp (nullable = true)
 |-- is_active: integer (nullable = true)
 |-- created_at: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- source_ingestion_ts: timestamp (nullable = true)
 |-- source_batch_id: string (nullable = true)
 |-- scd_hash: string (nullable = true)



## 5. Verificăm dacă tabela Silver există deja

In [0]:
table_exists = spark.catalog.tableExists(TARGET_TABLE)
print(f"Tabela {TARGET_TABLE} exista: {table_exists}")

if table_exists:
    existing_count = spark.table(TARGET_TABLE).count()
    print(f"Inregistrari existente: {existing_count}")

Tabela banking.silver.dim_customers exista: False


## 6A. Prima rulare — INSERT inițial

Dacă tabela nu există, e prima rulare. Inserăm toate rândurile cu:
- `customer_sk` generat (surrogate key)
- `valid_from = silver_loaded_at` (acum)
- `valid_to = NULL`
- `is_current = true`

In [0]:
if not table_exists:
    print("PRIMA RULARE — INSERT initial")
    
    now = current_timestamp()
    
    df_initial = (df_valid
        .withColumn("valid_from",        to_timestamp(lit("1900-01-01 00:00:00")))
        .withColumn("valid_to",          to_timestamp(lit("3000-01-01 00:00:00")))
        .withColumn("is_current",        lit(True))
        .withColumn("silver_loaded_at",  now)
    )
    
    # Generam customer_sk — auto-increment pornind de la 1
    window = Window.orderBy("customer_id")
    df_initial = df_initial.withColumn("customer_sk", row_number().over(window))
    
    # Reordonam coloanele: customer_sk primul (PK Silver)
    cols_order = ["customer_sk"] + [c for c in df_initial.columns if c != "customer_sk"]
    df_initial = df_initial.select(*cols_order)
    
    (df_initial.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TARGET_TABLE))
    
    print(f"  Inserat {df_initial.count()} versiuni initiale")
    print(f"  Toate cu is_current=true, valid_to=NULL")
else:
    print("Nu e prima rulare — sarim peste 6A si mergem la 6B")

PRIMA RULARE — INSERT initial


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  Inserat 500 versiuni initiale
  Toate cu is_current=true, valid_to=NULL


## 6B. Rulări următoare — MERGE cu detectare modificări

Algoritm:
1. Identificăm clienții **noi** (nu există în Silver)
2. Identificăm clienții **modificați** (hash diferit față de versiunea curentă)
3. Identificăm clienții **dispăruți** din Bronze (existau, nu mai sunt) → soft close
4. Pentru modificări:
   - Închidem versiunea curentă (`valid_to = now, is_current = false`)
   - Inserăm versiunea nouă (`valid_from = now, is_current = true`)
5. Pentru clienți noi: INSERT direct cu `is_current = true`

In [0]:
if table_exists:
    print("RULARE INCREMENTALA — MERGE cu detectare modificari")
    
    silver = spark.table(TARGET_TABLE)
    
    # Inregistrarile curente din Silver (versiunile active)
    silver_current = silver.filter(col("is_current") == True)
    
    # Bronze nou (cu hash deja calculat)
    bronze_new = df_valid.alias("b")
    
    # JOIN intre Bronze nou si Silver curent pe customer_id
    joined = bronze_new.join(
        silver_current.select(
            col("customer_id").alias("s_customer_id"),
            col("scd_hash").alias("s_scd_hash"),
            col("customer_sk").alias("s_customer_sk")
        ),
        col("b.customer_id") == col("s_customer_id"),
        "left_outer"
    )
    
    # Categorisim:
    # - NEW:        nu exista in Silver  →  s_customer_id IS NULL
    # - CHANGED:    exista, hash diferit →  s_customer_id NOT NULL si scd_hash != s_scd_hash
    # - UNCHANGED:  exista, hash identic →  ignoram
    
    df_new     = joined.filter(col("s_customer_id").isNull()).drop("s_customer_id", "s_scd_hash", "s_customer_sk")
    df_changed = joined.filter((col("s_customer_id").isNotNull()) & (col("scd_hash") != col("s_scd_hash")))
    
    n_new       = df_new.count()
    n_changed   = df_changed.count()
    n_unchanged = bronze_new.count() - n_new - n_changed
    
    print(f"  Clienti noi:       {n_new}")
    print(f"  Clienti modificati: {n_changed}")
    print(f"  Clienti neschimbati: {n_unchanged}")
    
    # ──────────────────────────────────────────
    # Pasul 1: Inchidem versiunile curente pentru clientii MODIFICATI
    # ──────────────────────────────────────────
    if n_changed > 0:
        ids_changed = [r[0] for r in df_changed.select("b.customer_id").collect()]
        
        delta_table = DeltaTable.forName(spark, TARGET_TABLE)
        delta_table.update(
            condition = (
                col("customer_id").isin(ids_changed) &
                (col("is_current") == True)
            ),
            set = {
                "valid_to":   "current_timestamp()",
                "is_current": "false",
            }
        )
        print(f"  Inchise {n_changed} versiuni vechi")
    
    # ──────────────────────────────────────────
    # Pasul 2: Inseram versiunile noi (atat NEW cat si CHANGED)
    # ──────────────────────────────────────────
    df_to_insert = df_new.select(*df_valid.columns).unionByName(
        df_changed.select(*df_valid.columns)
    ) if n_changed > 0 else df_new.select(*df_valid.columns)
    
    if df_to_insert.count() > 0:
        # Generam customer_sk — incepem de la max existent + 1
        max_sk = silver.agg(spark_max("customer_sk")).collect()[0][0] or 0
        
        now = current_timestamp()
        df_inserts = (df_to_insert
            .withColumn("valid_from",       now)
            .withColumn("valid_to",         to_timestamp(lit("3000-01-01 00:00:00")))
            .withColumn("is_current",       lit(True))
            .withColumn("silver_loaded_at", now)
        )
        
        # Generam customer_sk auto-incrementat
        window = Window.orderBy("customer_id")
        df_inserts = df_inserts.withColumn(
            "customer_sk",
            row_number().over(window) + lit(max_sk)
        )
        
        # Reordonam ca sa potriveasca schema target
        target_cols = spark.table(TARGET_TABLE).columns
        df_inserts = df_inserts.select(*target_cols)
        
        df_inserts.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
        
        print(f"  Inserat {df_inserts.count()} versiuni noi")
    
    # ──────────────────────────────────────────
    # Pasul 3: Soft-delete pentru clienti DISPARUTI din Bronze
    # ──────────────────────────────────────────
    bronze_ids = [r[0] for r in df_valid.select("customer_id").distinct().collect()]
    silver_current_ids = [r[0] for r in silver_current.select("customer_id").collect()]
    disappeared = list(set(silver_current_ids) - set(bronze_ids))
    
    if len(disappeared) > 0:
        delta_table = DeltaTable.forName(spark, TARGET_TABLE)
        delta_table.update(
            condition = (
                col("customer_id").isin(disappeared) &
                (col("is_current") == True)
            ),
            set = {
                "valid_to":   "current_timestamp()",
                "is_current": "false",
            }
        )
        print(f"  Soft-deleted {len(disappeared)} clienti disparuti din Bronze")
    else:
        print(f"  Niciun client disparut din Bronze")

## 7. Verificare — sumar dim_customers

In [0]:
%sql
SELECT 
    COUNT(*)                                AS total_versiuni,
    COUNT(DISTINCT customer_id)             AS clienti_unici,
    SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS versiuni_curente,
    SUM(CASE WHEN NOT is_current THEN 1 ELSE 0 END) AS versiuni_istorice,
    MIN(valid_from)                         AS prima_versiune,
    MAX(valid_from)                         AS ultima_versiune
FROM banking.silver.dim_customers;

total_versiuni,clienti_unici,versiuni_curente,versiuni_istorice,prima_versiune,ultima_versiune
500,500,500,0,2026-05-01T19:55:14.669Z,2026-05-01T19:55:14.669Z


## 8. Inspecție — clienți cu mai multe versiuni (modificări detectate)

In [0]:
%sql
-- Clienti care au mai mult de 1 versiune (au fost modificati cel putin o data)
WITH multi_version AS (
    SELECT customer_id, COUNT(*) AS n_versions
    FROM banking.silver.dim_customers
    GROUP BY customer_id
    HAVING COUNT(*) > 1
)
SELECT 
    d.customer_id,
    d.customer_sk,
    d.segment_code,
    d.kyc_status,
    d.county_code,
    d.is_active,
    d.valid_from,
    d.valid_to,
    d.is_current
FROM banking.silver.dim_customers d
INNER JOIN multi_version m ON d.customer_id = m.customer_id
ORDER BY d.customer_id, d.valid_from
LIMIT 30;

customer_id,customer_sk,segment_code,kyc_status,county_code,is_active,valid_from,valid_to,is_current


## 9. Inspecție — distribuția versiunilor per client

In [0]:
%sql
SELECT 
    n_versions, 
    COUNT(*) AS clienti_cu_acest_numar_versiuni
FROM (
    SELECT customer_id, COUNT(*) AS n_versions
    FROM banking.silver.dim_customers
    GROUP BY customer_id
)
GROUP BY n_versions
ORDER BY n_versions;

n_versions,clienti_cu_acest_numar_versiuni
1,500


## 10. Validare consistență SCD Type 2

Verificăm că **fiecare client are exact o versiune curentă** și că `is_current = (valid_to IS NULL)` este consistent.

In [0]:
%sql
-- Test 1: Fiecare customer_id are exact 1 versiune curenta
SELECT 
    COUNT(*) AS clienti_cu_multiple_versiuni_curente
FROM (
    SELECT customer_id
    FROM banking.silver.dim_customers
    WHERE is_current = true
    GROUP BY customer_id
    HAVING COUNT(*) > 1
);
-- Trebuie sa fie 0!

clienti_cu_multiple_versiuni_curente
0


In [0]:
%sql
-- Test 2: is_current si valid_to sunt consistente
SELECT 
    SUM(CASE WHEN is_current = true  AND valid_to IS NOT NULL THEN 1 ELSE 0 END) AS inconsistent_current,
    SUM(CASE WHEN is_current = false AND valid_to IS NULL     THEN 1 ELSE 0 END) AS inconsistent_historic
FROM banking.silver.dim_customers;
-- Ambele trebuie sa fie 0!

inconsistent_current,inconsistent_historic
0,0


## Sumar 02b2 — `dim_customers` cu SCD Type 2

Realizat:
- Validări complete (CNP format, email, FK-uri către ref_*)
- Hash SHA-256 al atributelor SCD-tracked pentru detectare modificări
- Logică dual-mode: prima rulare = INSERT, rulări ulterioare = MERGE incremental
- Surrogate key `customer_sk` auto-incrementat
- Coloane SCD: `valid_from`, `valid_to`, `is_current` (Kimball pattern)
- Soft delete pentru clienți dispăruți din Bronze
- Quarantine pentru rânduri invalide cu `error_reasons` array
- Validări de consistență la final

**Următorul pas:** `02c_silver_facts` — `fact_transactions` + `dim_date`. Acolo:
- Validări multiple pe tranzacții (8% în quarantine)
- Deduplicare pe `transaction_id` (păstrăm cea mai recentă)
- JOIN-uri către dimensiuni pentru a obține surrogate keys
- Generare `dim_date` static pe 5 ani

In [0]:
%sql
UPDATE banking.silver.dim_customers
SET 
    valid_from = TIMESTAMP '1900-01-01 00:00:00',
    valid_to   = TIMESTAMP '3000-01-01 00:00:00'
WHERE is_current = true;

-- Verificare
SELECT 
    MIN(valid_from) AS min_from, 
    MAX(valid_from) AS max_from,
    MIN(valid_to)   AS min_to,
    MAX(valid_to)   AS max_to,
    SUM(CASE WHEN valid_to IS NULL THEN 1 ELSE 0 END) AS still_null
FROM banking.silver.dim_customers
WHERE is_current = true;
-- Trebuie: min_from = 1900-01-01, max_to = 3000-01-01, still_null = 0

min_from,max_from,min_to,max_to,still_null
1900-01-01T00:00:00.000Z,1900-01-01T00:00:00.000Z,3000-01-01T00:00:00.000Z,3000-01-01T00:00:00.000Z,0
